In [ ]:
!pip install pandas supabase

In [ ]:
import pandas as pd
from supabase import create_client, Client
import os

# ==========================================
# Configuration Variables
# ==========================================
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
CSV_FILENAME = "tournament_players.csv"
TABLE_NAME = "trainers" # Adjust this if your target table is named differently

def process_and_upsert_players(csv_path: str, url: str, key: str, table: str):
    """
    Reads a CSV, deduplicates based on player ID keeping the latest entry,
    formats the data, and upserts it into a Supabase table.
    """
    print(f"Reading data from {csv_path}...")
    
    # 1. Load the CSV data
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: Could not find the file '{csv_path}'.")
        return

    # 2. Deduplicate the entries
    # Since the CSV is pre-sorted by date, the 'last' occurrence is the newest.
    initial_count = len(df)
    df_deduped = df.drop_duplicates(subset=['Dropdown Text'], keep='last')
    print(f"Removed {initial_count - len(df_deduped)} duplicate rows.")

    # 3. Filter out the specific columns we need and rename them to match the database schema
    df_mapped = df_deduped[['Dropdown Text', 'Player Name']].copy()
    df_mapped.rename(columns={
        'Dropdown Text': 'id',
        'Player Name': 'alphabet_name'
    }, inplace=True)

    # 4. Convert the DataFrame into a list of dictionaries for the Supabase client
    records_to_upsert = df_mapped.to_dict(orient='records')

    # 5. Initialize Supabase client and perform the upsert
    print(f"Connecting to Supabase and upserting {len(records_to_upsert)} records...")
    try:
        supabase: Client = create_client(url, key)
        
        # The upsert method automatically inserts new rows or updates existing ones 
        # based on the primary key ('id')
        response = supabase.table(table).upsert(records_to_upsert).execute()
        
        print("Upsert completed successfully!")
        
        # Displaying a preview of the processed data in the notebook
        return pd.DataFrame(response.data).head()
        
    except Exception as e:
        print(f"An error occurred during the Supabase operation: {e}")


# ==========================================
# Execution
# ==========================================
# Run the function using the configuration variables above
uploaded_data_preview = process_and_upsert_players(
    csv_path=CSV_FILENAME, 
    url=SUPABASE_URL, 
    key=SUPABASE_KEY, 
    table=TABLE_NAME
)

# Render the preview in Jupyter
uploaded_data_preview